# Ch.3 画像処理入門（Pillow × NumPy）

## 本チャプターのゴール
- 画像データの正体（ピクセル値の2次元配列）を理解する
- Pillow で画像の基本操作（リサイズ・変換・フィルタ）ができる
- 画像データを数値として分析できる

## 使用データ
scikit-learnに内蔵されている `digits` データセット
- 手書き数字（0〜9）の画像：1,797枚
- 各画像は **8×8ピクセル**、グレースケール（0〜16の値）
- 実務の画像処理（製造業の外観検査、医療画像分析）のミニチュア版

## 実務ユースケース
| 業界 | 用途 |
|------|------|
| 製造業 | 製品の傷・欠陥の自動検出 |
| 医療 | レントゲン・MRI画像の解析 |
| 小売 | 商品の棚卸し・在庫確認 |
| 金融 | 書類・帳票の文字認識（OCR） |

---

## 🤖 生成AIの使い方ガイド（本チャプター）

| AIに任せてよいこと | 自分で理解すること |
|---|---|
| `ImageFilter` のオプション一覧の調べ方 | グレースケール変換で何の情報が失われるか |
| `subplot` で画像を並べる方法 | ピクセル値の平均が何を意味するか |
| PIL Image から numpy array への変換 | フィルタ処理の前後でどう変化したか |


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image, ImageFilter, ImageEnhance
from sklearn.datasets import load_digits
import japanize_matplotlib

# digits データセットの読み込み
digits = load_digits()

print('画像データの形状:', digits.images.shape)
print('  → {}枚の画像、各画像は{}×{}ピクセル'.format(*digits.images.shape))
print('ラベル（数字）の種類:', np.unique(digits.target))

---
## 3.1 画像データの正体：数値の配列

In [ ]:
# 1枚目の画像データ（数字「0」）を確認
img_array = digits.images[0]
print('画像データ（8×8の数値配列）：')
print(img_array.astype(int))  # 0〜16の整数値
print('\nラベル（正解の数字）:', digits.target[0])

In [ ]:
# 数字を視覚的に確認
fig, axes = plt.subplots(2, 5, figsize=(12, 5))

for i, ax in enumerate(axes.flat):
    ax.imshow(digits.images[i], cmap='gray_r')  # gray_r = 黒背景に白い数字
    ax.set_title(f'ラベル: {digits.target[i]}')
    ax.axis('off')

plt.suptitle('手書き数字データセット（最初の10枚）', fontsize=14)
plt.tight_layout()
plt.show()

---
## 3.2 Pillow（PIL）で画像を操作する

numpy array → PIL Image → 各種操作

In [ ]:
# numpy array から PIL Image に変換
# digits の値は 0〜16 → 0〜255 にスケール変換
img_array_scaled = (digits.images[0] / 16.0 * 255).astype(np.uint8)

img_pil = Image.fromarray(img_array_scaled, mode='L')  # L = グレースケール
print('PIL Image サイズ:', img_pil.size)  # (幅, 高さ)
print('PIL Image モード:', img_pil.mode)  # L = グレースケール

# 表示
plt.figure(figsize=(3, 3))
plt.imshow(img_pil, cmap='gray_r')
plt.title('元画像（8×8ピクセル）')
plt.axis('off')
plt.show()

In [ ]:
# リサイズ：8×8 → 80×80（10倍に拡大）
img_large = img_pil.resize((80, 80), Image.NEAREST)  # NEAREST = 最近傍補間

fig, axes = plt.subplots(1, 2, figsize=(8, 4))
axes[0].imshow(img_pil, cmap='gray_r')
axes[0].set_title(f'元画像: {img_pil.size}')
axes[0].axis('off')

axes[1].imshow(img_large, cmap='gray_r')
axes[1].set_title(f'リサイズ後: {img_large.size}')
axes[1].axis('off')

plt.suptitle('リサイズ（8×8 → 80×80）')
plt.tight_layout()
plt.show()

---
## 3.3 フィルタ処理：画像を変換する

In [ ]:
# 作業用：80×80に拡大した画像を準備
def prepare_digit_image(digit_idx):
    """指定インデックスの digits 画像を 80×80 の PIL Image として返す"""
    arr = (digits.images[digit_idx] / 16.0 * 255).astype(np.uint8)
    return Image.fromarray(arr, mode='L').resize((80, 80), Image.NEAREST)

# 数字「8」の画像で試す
idx_8 = np.where(digits.target == 8)[0][0]  # 最初の「8」のインデックス
img_base = prepare_digit_image(idx_8)

# 各種フィルタを適用
img_blur    = img_base.filter(ImageFilter.BLUR)       # ぼかし
img_sharp   = img_base.filter(ImageFilter.SHARPEN)    # シャープ化
img_contour = img_base.filter(ImageFilter.CONTOUR)    # 輪郭抽出
img_emboss  = img_base.filter(ImageFilter.EMBOSS)     # エンボス

# 並べて表示
fig, axes = plt.subplots(1, 5, figsize=(15, 3))
images = [img_base, img_blur, img_sharp, img_contour, img_emboss]
titles = ['元画像', 'BLUR（ぼかし）', 'SHARPEN（シャープ）', 'CONTOUR（輪郭）', 'EMBOSS（浮き彫り）']

for ax, img, title in zip(axes, images, titles):
    ax.imshow(img, cmap='gray_r')
    ax.set_title(title, fontsize=10)
    ax.axis('off')

plt.suptitle(f'フィルタ処理の比較（数字: {digits.target[idx_8]}）', fontsize=13)
plt.tight_layout()
plt.show()

---
## 3.4 ピクセル値の統計：「明るさ」を数値で分析する

画像 = 数値の配列 なので、統計量も計算できます

In [ ]:
# 画像を numpy array に戻して統計計算
arr = np.array(img_base)

print('ピクセル値の統計（数字「8」）:')
print(f'  平均（明るさ）: {arr.mean():.1f}')
print(f'  最大値:        {arr.max()}')
print(f'  最小値:        {arr.min()}')
print(f'  標準偏差:      {arr.std():.1f}')

In [ ]:
# 数字（0〜9）ごとの「平均ピクセル値（明るさ）」を比較
brightness_by_digit = {}

for digit in range(10):
    indices = np.where(digits.target == digit)[0]
    # 各数字の全画像の平均ピクセル値
    mean_brightness = digits.images[indices].mean()
    brightness_by_digit[digit] = mean_brightness

# 棒グラフで表示
fig, ax = plt.subplots(figsize=(9, 4))
ax.bar(brightness_by_digit.keys(), brightness_by_digit.values(),
       color='steelblue', edgecolor='white')
ax.set_title('数字ごとの平均ピクセル値（明るさ）')
ax.set_xlabel('数字（ラベル）')
ax.set_ylabel('平均ピクセル値（0〜16）')
ax.set_xticks(range(10))
plt.tight_layout()
plt.show()

# 観察：数字「1」はピクセルが少ないので暗く（値が低い）なる

---
## 🖊️ 演習（40分）

---

### 問1：好きな数字（0〜9）を選んで、フィルタ処理の前後を並べて表示する

In [ ]:
# TODO: 好きな数字を選び、元画像と CONTOUR フィルタ後の画像を横並びで表示してください
# ヒント：prepare_digit_image(idx) 関数を使います
#         数字「3」の最初のインデックスは np.where(digits.target == 3)[0][0] で取得できます

target_digit = 3  # ← ここを変えてください（0〜9）



### 問2：同じ数字（例：「3」）の複数枚の画像を並べて表示する

手書きは人によってバラバラ → それがAIにとって難しい理由を体感する

In [ ]:
# TODO: 数字「3」のデータを最低5枚取り出して横並びで表示してください
# ヒント：np.where(digits.target == 3)[0] でインデックス一覧が取れます



### 問3：フィルタ処理の前後でピクセル値の統計がどう変わるか確認する

In [ ]:
# TODO: 任意の画像に BLUR と CONTOUR を適用し、
#       元画像・BLUR後・CONTOUR後のそれぞれのピクセル値の平均・標準偏差を表示してください
#
# 結果を見て：フィルタによって数値がどう変化するか、理由を考えてみましょう



### 問4（発展）：明るさによる2値化（閾値処理）

一定のピクセル値より明るければ白（255）、暗ければ黒（0）に変換する処理を実装する

In [ ]:
# TODO: numpy を使って、ピクセル値が 128 以上なら 255（白）、
#       未満なら 0（黒）に変換した2値化画像を作り、元画像と並べて表示してください
# ヒント：np.where(条件, true時の値, false時の値) または arr > 128 を使います



---
## ✅ チャプターのまとめ

| 操作 | コード |
|------|--------|
| numpy → PIL Image | `Image.fromarray(arr, mode='L')` |
| PIL Image → numpy | `np.array(img)` |
| リサイズ | `img.resize((幅, 高さ), Image.NEAREST)` |
| フィルタ適用 | `img.filter(ImageFilter.BLUR)` |
| ピクセル値の統計 | `np.array(img).mean()` など |

**重要な概念**
- 画像 = 数値の2次元配列
- ピクセル値が小さい → 暗い（黒に近い）
- グレースケール = 色情報（RGB）を失い、明るさだけ残す
- フィルタ = 周辺ピクセルの値を使って計算する演算